In [ ]:
# =============================================================================
# FUMD-AI Training & Postprocessing Workflow -- Step 1: Load & window dataset
# =============================================================================
# Step:         1 of 6
# Summary:      Load a dataset_labeled_w<W>.csv, encode categoricals, split by vehicle, build sliding-window sequences, fit/apply the feature scaler.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-training-postprocessing-workflow
# Version:      0.1.1
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# =============================================================================

# Step 1 — Load & window dataset

Part of the **FUMD-AI training & postprocessing workflow** (step 1 of 6).

**Purpose.** Turn one AI-ready `dataset_labeled_w<W>.csv` (the output of the
[FUMD-AI preprocessing workflow](https://github.com/FUMD-AI/fumd-ai-preprocessing-workflow)'s
Step 6) into the scaled sliding-window arrays Step 2 trains on:

1. Load the dataset and defensively normalize it to one row per vehicle per
   second (the preprocessing workflow's output already is, so this is
   normally a no-op).
2. Fit label encoders for the one categorical column (`lane`, a per-map
   SUMO road-edge id) and encode it.
3. Split **by vehicle** (not by window) into train/validation, so
   validation never shares a source row with a training window (see
   `data.split_vehicles`'s docstring for why this matters).
4. Build `(X, y)` sliding windows per split (`data.build_sequences`):
   `SEQUENCE_LENGTH` seconds of history -> `servingCell` at each of
   `FUTURE_STEPS` seconds ahead. `migration`/`destination` are carried
   through per-window for Step 3's evaluation breakdown, never as features
   (see `data.py`'s module docstring for why `servingCell1..7`,
   `migration`, and `destination` are all excluded from the feature set).
5. Fit a `StandardScaler` on the **training** windows only, and apply it
   (transform-only) to both splits.

**Input:** `DATASET_PATH` — defaults to the bundled tiny example.

**Outputs**, written under `OUTPUT_DIR`:
- `windows.npz` — `X_train`, `y_train`, `X_val`, `y_val`, plus per-window
  `migration_train`/`migration_val`/`destination_train`/`destination_val`
- `scaler.joblib`, `label_encoders.joblib`
- `run_manifest.json` — feature list, sequence_length, future_steps,
  num_base_stations, source dataset path (Step 2 appends model
  hyperparameters to the same file)


In [ ]:
import os
import sys

import joblib
import numpy as np

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # `jupyter execute` sets the kernel's cwd to this notebook's OWN
                     # directory (see nbclient's cli.py), not the repo root -- normalize
                     # back so `sys.path.insert(0, "src")` and every relative default
                     # below resolve the same way whether this notebook is run standalone
                     # (jupyter execute notebooks/step_N...ipynb, e.g. via slurm/*.sbatch)
                     # or chained by run_pipeline.ipynb (whose own cwd is already the repo
                     # root, and which additionally sets cwd explicitly -- see
                     # fumd_training_workflow.notebook_runner.run_step).

sys.path.insert(0, "src")  # so `import fumd_training_workflow` finds src/fumd_training_workflow/
from fumd_training_workflow import data as d
from fumd_training_workflow.evaluate import infer_num_base_stations
from fumd_training_workflow.metrics_io import save_run_manifest

In [ ]:
# ---- Parameters ----
# This cell is tagged "parameters" so the notebook can also be executed
# headlessly (see notebook_runner.run_step / run_pipeline.ipynb), or edited
# by hand for a standalone `jupyter execute` / Slurm submission.

# Default points at the tiny bundled example so this notebook runs out of
# the box -- replace with your own dataset_labeled_w<W>.csv from the
# preprocessing workflow directly.
DATASET_PATH = "example-data/dataset_labeled_example.csv"

# None -> data.DEFAULT_FEATURE_COLUMNS (everything except the lead/leakage
# and label-only columns -- see data.py's module docstring).
FEATURE_COLUMNS = None

SEQUENCE_LENGTH = 6          # seconds of history per window
FUTURE_STEPS = [1, 2, 3, 4, 5, 6, 7]  # seconds ahead to forecast
VAL_FRACTION = 0.2           # fraction of *vehicles* held out for validation
RANDOM_SEED = 42

OUTPUT_DIR = "pipeline_run"  # where windows.npz / scaler / encoders / manifest are written

In [ ]:
# ---- Environment variable overrides (optional) ----
# `jupyter execute` has no -p/parameter-injection flag (unlike papermill), so
# each parameter above can also be set via an environment variable of the same
# name instead of hand-editing this cell, e.g.:
#   DATASET_PATH=/abs/path/dataset_labeled_w3.csv OUTPUT_DIR=/abs/path/pipeline_run \
#   jupyter execute notebooks/step_1_load_and_window_dataset.ipynb
# See fumd_training_workflow.notebook_runner.env_override's docstring for exact
# value-parsing rules. Unset variables leave the parameters cell's own default above untouched.
from fumd_training_workflow.notebook_runner import env_override

DATASET_PATH = env_override('DATASET_PATH', DATASET_PATH)
FEATURE_COLUMNS = env_override('FEATURE_COLUMNS', FEATURE_COLUMNS)
SEQUENCE_LENGTH = env_override('SEQUENCE_LENGTH', SEQUENCE_LENGTH)
FUTURE_STEPS = env_override('FUTURE_STEPS', FUTURE_STEPS)
VAL_FRACTION = env_override('VAL_FRACTION', VAL_FRACTION)
RANDOM_SEED = env_override('RANDOM_SEED', RANDOM_SEED)
OUTPUT_DIR = env_override('OUTPUT_DIR', OUTPUT_DIR)

## 1. Load and normalize

In [ ]:
feature_columns = FEATURE_COLUMNS or d.DEFAULT_FEATURE_COLUMNS

df = d.load_labeled_dataset(DATASET_PATH)
print("loaded:", df.shape)

df = d.ensure_one_row_per_vehicle_second(df)
print("after resample-guard:", df.shape)

num_base_stations = infer_num_base_stations(df)
print("num_base_stations (derived from data):", num_base_stations)

## 2. Encode categoricals

Fit once, on the whole dataset before splitting -- `lane` is a per-map road-edge id, and every value that appears anywhere in this run should get a stable code (not just the ones that happen to land in the training split).

In [ ]:
label_encoders = d.fit_label_encoders(df)
df = d.apply_label_encoders(df, label_encoders)
for col, enc in label_encoders.items():
    print(f"{col}: {len(enc.classes_)} classes (+ 1 reserved 'unknown' code)")

## 3. Split by vehicle

In [ ]:
train_df, val_df = d.split_vehicles(df, val_fraction=VAL_FRACTION, seed=RANDOM_SEED)

## 4. Build sliding windows

In [ ]:
X_train, y_train, extra_train = d.build_sequences(
    train_df, feature_columns=feature_columns, sequence_length=SEQUENCE_LENGTH, future_steps=FUTURE_STEPS,
)
X_val, y_val, extra_val = d.build_sequences(
    val_df, feature_columns=feature_columns, sequence_length=SEQUENCE_LENGTH, future_steps=FUTURE_STEPS,
)

## 5. Fit & apply the scaler (train-only)

In [ ]:
scaler = d.fit_scaler(X_train)
X_train_scaled = d.apply_scaler(scaler, X_train)
X_val_scaled = d.apply_scaler(scaler, X_val)  # transform only -- never re-fit on validation data

## 6. Save

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.savez_compressed(
    os.path.join(OUTPUT_DIR, "windows.npz"),
    X_train=X_train_scaled, y_train=y_train,
    X_val=X_val_scaled, y_val=y_val,
    migration_train=extra_train["migration"], destination_train=extra_train["destination"],
    migration_val=extra_val["migration"], destination_val=extra_val["destination"],
)
joblib.dump(scaler, os.path.join(OUTPUT_DIR, "scaler.joblib"))
joblib.dump(label_encoders, os.path.join(OUTPUT_DIR, "label_encoders.joblib"))

manifest = {
    "dataset_path": DATASET_PATH,
    "feature_columns": feature_columns,
    "sequence_length": SEQUENCE_LENGTH,
    "future_steps": FUTURE_STEPS,
    "num_base_stations": num_base_stations,
    "val_fraction": VAL_FRACTION,
    "random_seed": RANDOM_SEED,
    "n_train_windows": int(X_train_scaled.shape[0]),
    "n_val_windows": int(X_val_scaled.shape[0]),
}
save_run_manifest(OUTPUT_DIR, manifest)

print(f"saved windows.npz, scaler.joblib, label_encoders.joblib, run_manifest.json -> {OUTPUT_DIR}")
print(f"train windows: {X_train_scaled.shape}, val windows: {X_val_scaled.shape}")